# 06. 조건으로 값을 고르고 방향별로 요약하기

먼저 결과를 예상하고, 코드를 직접 실행한 뒤 실제 값과 결과 모양을 확인함. 이 파일은 위에서 아래로 차례로 실행하면 됨.


### 먼저 알아둘 배열 말 네 가지

`np.array([[1, 2], [3, 4]])`는 NumPy가 다루는 2차원 숫자 배열임. NumPy의 정확한 이름은 **ndarray**이며, 같은 저장 형식의 값을 여러 축에 놓는 자료구조임.

- **shape(결과 모양)**: 각 축의 칸 수임. 위 배열은 2행 2열이라 `(2, 2)`이며 `a.shape`로 확인함.
- **dimension(차원)**: 축의 개수임. 위 배열은 축이 두 개라 `a.ndim == 2`임. 차원과 전체 원소 수는 다름.
- **axis(축)**: 값의 위치가 변하는 방향임. 2차원 표에서 `axis=0`은 행을 따라 내려가는 방향, `axis=1`은 열을 따라 옆으로 가는 방향임. 축 번호만으로 실제 센서 의미나 단위까지 정해지지는 않음.
- **dtype(저장 형식)**: 각 값을 정수·실수·문자 중 어떤 방식으로 저장하는지 나타냄. `a.dtype`으로 확인하며, 값의 단위나 현실 의미를 보장하지 않음.

계산에서 **숫자 하나로 다루는 값**을 scalar(스칼라)라고 부름. 다만 Python 숫자 `10`은 `.shape`가 없고, `np.array(10)`은 shape `()`인 0차원 ndarray이며, `np.float64(10)` 같은 NumPy scalar도 별도 타입이라 세 표현이 완전히 같지는 않음. `np.asarray(value).shape`로 배열 관점의 모양을 확인함. `True`와 `False`는 **Boolean(참·거짓)** 값임. Boolean 배열을 **mask(마스크)**로 쓰면 `True`인 위치만 고를 수 있지만, mask의 모양과 어느 축을 고르는지는 먼저 확인해야 함.

먼저 결과를 예측하고, 코드를 직접 실행한 뒤 실제 값과 모양(shape)을 확인함.


In [ ]:
import json
import numpy as np

오류_예시를_실행할지 = False
연습_데이터 = json.loads(r'''{"mask_axis":{"data":[[68,0.31,101.2],[74,0.48,99.8],[76,0.52,103.1],[81,0.61,104],[72,0.55,98.9],[77,0.44,102.2]],"element_rule":{"op":">","per_column_threshold":[75,0.5,103]}}}''')

def 배열_확인(name, value):
    array = np.asarray(value)
    print(f"{name}: value={array}, shape={array.shape}, dtype={array.dtype}")
    return array

print(f"NumPy version: {np.__version__}")
if 연습_데이터:
    print(f"이 단원에서 쓸 연습 데이터: {list(연습_데이터)}")


## 조건으로 값을 고르고 방향별로 요약하기

**핵심 질문:** 조건에 맞는 행 또는 원소를 고르고 요약할 때 mask와 axis를 어떻게 검증할까?


#### 먼저 생각

**조건 선택 용어부터 확인함.** Boolean은 `True/False`, mask는 이를 배열로 모은 선택표임. **feature(특성)**은 샘플 하나를 설명하는 값 한 종류이며, `[[70, .4], [80, .6]]`에서는 각 열 하나가 특성 하나임. `a=np.array([10,20,30]); a[np.array([True,False,True])]`는 `[10,30]`을 만듦. 같은 shape의 mask는 원소를 고르고, 한 축 길이와 같은 mask는 그 축의 행이나 열을 고를 수 있으므로 결과 모양을 먼저 확인함. 열을 특성이라고 부른다고 해서 이름·단위·현실 의미까지 shape가 보장하는 것은 아님.
**핵심 질문**: 조건에 맞는 행 또는 원소를 고르고 요약할 때 mask와 axis를 어떻게 검증할까?

**실행 전 예측**

(6,3) 데이터에 (6,) 행 mask와 (6,3) 원소 mask를 적용했을 때 결과 shape를 예측하셈.

> 내 예측(값·shape·조건·단위): `TODO`


#### 개념

**왜 배우는지와 주의할 점**

mask는 고를 자리를 True, 버릴 자리를 False로 적은 배열임. `(m,)` mask는 axis 0의 행을, (m,n) mask는 개별 원소를 고를 수 있음. 복합 배열 조건은 괄호와 `&`·`|`를 씀. 기본 reduction은 지정 축을 제거함.

코드를 볼 때는 무엇을 계산하는지, 결과 모양이 어떤지, 원래 배열이 바뀌는지를 함께 확인함.


### 개념 도식




In [ ]:
raw = np.array(연습_데이터['mask_axis']['data'], dtype=float)
row_mask = (raw[:,0] >= 75) & (raw[:,1] >= .5)
element_threshold = np.array(연습_데이터['mask_axis']['element_rule']['per_column_threshold'])
element_mask = raw > element_threshold
print(f"row/element selections: {raw[row_mask].shape} {raw[element_mask].shape}")
print(f"axis means: {raw.mean(axis=0).shape} {raw.mean(axis=1).shape}")
if 오류_예시를_실행할지:
    _ = (raw[:,0] >= 75) and (raw[:,1] >= .5)


### 실행 뒤 해석

방금 출력에서 실제 값, 결과 모양(shape), 저장 형식(dtype), 원래 배열이 바뀌었는지를 한 문장으로 정리하셈. 예상과 다르면 어느 입력이나 축을 다시 확인할지도 적으셈.


### 🧪 학생 문제

온도 78 이상이고 압력 103 이상인 행을 고르고 센서별 평균과 시점별 평균을 계산하셈. mask와 결과 shape부터 제안하셈.

1. 결과·shape·조건을 먼저 쓰셈.
2. 아래 TODO 셀에 자기 코드를 작성하셈.
3. 출력은 `실제 값 / 결과 모양(shape) / 저장 형식(dtype) / 오류`로 기록하셈.


In [ ]:
# 직접 작성할 부분
problem_raw = np.array([[70,.40,100],[76,.51,103],[79,.48,104],[82,.63,105]])
# TODO: 온도 78 이상이고 압력 103 이상인 row_mask부터 작성하고 selected, feature_mean, row_mean을 완성하셈.
row_mask = None
selected = None
feature_mean = None
row_mean = None
print(f"입력 shape: {problem_raw.shape}")


### 풀이 전 자기 점검

- 결과 shape가 예측과 같은가?
- 계산 목적과 연산이 일치하는가?
- 단위·원본 변경·경계 조건을 확인했는가?


### 다른 조건에서 교정·재시도

완성 답안을 복사하지 말고, 조건이 바뀐 입력에서 판단 절차를 다시 적용함. 아래 셀은 다른 입력의 안전한 실행 결과임.


In [ ]:
# 조건을 바꿔 다시 확인함
retry = np.array([[1, 10], [2, 20], [3, 30]])
retry_row_mask = retry[:, 0] >= 2
retry_element_mask = retry >= np.array([2, 20])
print(f"행 선택: {retry[retry_row_mask]} {retry[retry_row_mask].shape}")
print(f"원소 선택: {retry[retry_element_mask]} {retry[retry_element_mask].shape}")
print(f"keepdims 비교: {retry.mean(axis=0).shape} {retry.mean(axis=0, keepdims=True).shape}")


#### 응용

운영 규칙을 mask로 번역할 때 단위·대상 차원·선택 의미를 기록하고 위험도 판단으로 과장하지 않음.

> 새 조건에서 유지할 판단과 보류할 해석: `TODO`


### 자기 점검

- [ ] 실행 전 결과·shape·조건을 예측함.
- [ ] 실제 출력·shape·dtype 또는 오류를 기록함.
- [ ] 오류를 원인과 유형으로 설명함.
- [ ] 최소 수정 후 다른 조건에서 재실행함.
- [ ] 계산 가능성과 해석 가능성을 구분함.

내가 아직 증명하지 못한 것: `TODO`
